# fetch

> Download YouTube xscripts and metadata locally via yt-dlp

In [ ]:
#| default_exp fetch

## What to explore

- `yt-dlp` xscript download (subtitles / auto-generated captions)
- Metadata extraction (title, duration, channel, upload date)
- Storage format and directory layout for 11 Solveit videos

In [ ]:
#| eval: false
import yt_dlp
print(yt_dlp.version.__version__)

In [ ]:
#| eval: false
from pathlib import Path

links_path = Path('solveit2.txt')
urls = [link.strip() for link in links_path.read_text(encoding='utf-8').splitlines() if link.strip()]

len(urls), urls[5]

In [ ]:
#| eval: false
url = urls[5]
ydl_opts = {'skip_download': True, 'quiet': False}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

In [ ]:
#| eval: false
# Check the fields we care about
for k in ['id', 'title', 'channel', 'duration', 'upload_date', 'webpage_url', 'description']:
    print(f'{k}: {info.get(k)}')

In [ ]:
#| eval: false
# Check subtitle tracks available
print('subtitles (manual):', list(info.get('subtitles', {}).keys()))
print('automatic_captions:', list(info.get('automatic_captions', {}).keys())[:10])

In [ ]:
#| eval: false
# Is English auto-caption available?
auto = info.get('automatic_captions', {})
print('en' in auto)
# What formats are available for English?
if 'en' in auto:
    for entry in auto['en']:
        print(f"  ext={entry['ext']}")

In [ ]:
#| eval: false
# Check all 11 videos: manual subtitles vs auto captions
for u in urls:
    vid = u.split('/')[-1].split('=')[-1]
    opts = {'skip_download': True, 'quiet': True}
    with yt_dlp.YoutubeDL(opts) as ydl:
        i = ydl.extract_info(u, download=False)
    subs = [k for k in i.get('subtitles', {}) if k != 'live_chat']
    has_auto_en = 'en' in i.get('automatic_captions', {})
    print(f'{vid}: manual={subs or "none"}, auto_en={has_auto_en}')

## Compare caption formats: vtt vs srt vs json3

In [ ]:
#| eval: false
import tempfile, os

# Download English auto-captions in 3 formats for comparison
tmp = tempfile.mkdtemp()
for fmt in ['vtt', 'srt', 'json3']:
    opts = {
        'skip_download': True, 'quiet': True,
        'writeautomaticsub': True, 'subtitleslangs': ['en'],
        'subtitlesformat': fmt,
        'outtmpl': os.path.join(tmp, f'sample.%(ext)s'),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

# Show what was downloaded
for f in sorted(os.listdir(tmp)):
    sz = os.path.getsize(os.path.join(tmp, f))
    print(f'{f} ({sz:,} bytes)')

In [ ]:
#| eval: false
# Show first 20 lines of each format
import glob
for f in sorted(glob.glob(os.path.join(tmp, '*'))):
    print(f'=== {os.path.basename(f)} ===')
    with open(f) as fh:
        for i, line in enumerate(fh):
            if i >= 20: break
            print(line, end='')
    print('\n')

## Observations

**Caption availability (11 videos):**
- 2/11 have manual English subtitles (`7T83srD0Mu4`, `uSkO1BDfcNM`)
- 11/11 have auto-generated English captions
- `live_chat` appears as a subtitle track on live archives — not actual captions

**Format comparison:**

| Format | Size | Notes |
|--------|------|-------|
| json3 | 2.1MB | Complex header, hard to parse |
| vtt | 1.0MB | Word-level timing tags (`<c>`), HTML escapes |
| **srt** | **213KB** | **Simplest — sequential number + timestamp + text. Best for downstream parsing** |

**Metadata:** `id`, `title`, `channel`, `duration`, `upload_date`, `webpage_url` all available, including live archives.

**Live vs normal:** No special handling needed — live archives behave identically.

**yt-dlp warnings:** JS runtime (`deno`) and impersonation warnings appear but do not affect metadata or caption downloads. These only impact video format extraction which we don't use. May require attention if yt-dlp changes behavior in future versions.

## Storage design

**Storage location:** `~/.cache/yttoc/{video_id}/` (XDG Base Directory spec — re-downloadable data belongs in cache). Overridable via `--root`.

```
~/.cache/yttoc/
  {video_id}/
    metadata.json
    captions.en.srt        # manual if available, otherwise auto-generated
```

**metadata.json fields:**
```json
{
  "id": "Ap4DmaB0gLY",
  "title": "Sysadmin, devops, and web scraping - Solveit lesson 6",
  "channel": "Jeremy Howard",
  "duration": 8192,
  "upload_date": "20251117",
  "webpage_url": "https://www.youtube.com/watch?v=Ap4DmaB0gLY",
  "caption_type": "auto"
}
```

**Caption selection logic:** prefer manual English (`en`, then `en-*`) → fall back to auto English (`en`, then `en-*`) → error if neither exists.

**CLI:** `yttoc_fetch <url>` fetches one video. Batch via shell: `cat urls.txt | xargs -I{} yttoc_fetch {}`

One srt file per video. No need to store multiple formats or languages for now.

In [ ]:
#| hide
#| eval: false
import json
from pathlib import Path

# Try saving one video: metadata + srt caption
vid_id = info['id']
out_dir = Path('downloads') / vid_id
out_dir.mkdir(parents=True, exist_ok=True)

# Determine caption type
has_manual = 'en' in info.get('subtitles', {})
caption_type = 'manual' if has_manual else 'auto'

# Save metadata
meta = {
    'id': info['id'],
    'title': info['title'],
    'channel': info['channel'],
    'duration': info['duration'],
    'upload_date': info['upload_date'],
    'webpage_url': info['webpage_url'],
    'caption_type': caption_type,
    'series': 'solveit2',
    'series_index': urls.index(url),
}
(out_dir / 'metadata.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps(meta, indent=2))

In [ ]:
#| hide
#| eval: false
# Download srt caption to the same directory
sub_opt = 'writesubtitles' if has_manual else 'writeautomaticsub'
opts = {
    'skip_download': True, 'quiet': True,
    sub_opt: True,
    'subtitleslangs': ['en'],
    'subtitlesformat': 'srt',
    'outtmpl': str(out_dir / 'captions.%(ext)s'),
}
with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([url])

# Verify
for f in sorted(out_dir.iterdir()):
    print(f'{f.name} ({f.stat().st_size:,} bytes)')

## Modularize

In [ ]:
#| export
import json, os
from pathlib import Path
import yt_dlp

_DEFAULT_ROOT = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'yttoc'

In [ ]:
#| export
def load_urls(path: str | Path # Path to URL list file (e.g. 'nbs/solveit2.txt')
             ) -> tuple[list[str], str]: # (urls, series_name)
    "Load URLs and derive series name from filename."
    p = Path(path)
    urls = [l.strip() for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
    series = p.stem
    return urls, series

In [ ]:
# Test: load_urls
test_urls, test_series = load_urls('solveit2.txt')
assert len(test_urls) == 11
assert test_series == 'solveit2'
assert test_urls[0].startswith('https://')
print('ok')

In [ ]:
#| export
def _pick_lang(tracks: dict,
               base_lang: str = 'en' # Preferred base language
              ) -> str | None: # Best-matching language key
    "Select an exact or prefix language match from yt-dlp subtitle tracks."
    tracks = tracks or {}
    if base_lang in tracks: return base_lang
    for lang in sorted(k for k in tracks if k != 'live_chat'):
        if lang.startswith(f'{base_lang}-'):
            return lang
    return None

def _find_downloaded_srt(out_dir: str | Path # Directory containing downloaded captions
                        ) -> Path | None: # Single SRT path if present
    "Locate the downloaded SRT file emitted by yt-dlp."
    matches = sorted(Path(out_dir).glob('captions*.srt'))
    if not matches: return None
    if len(matches) > 1:
        names = ', '.join(p.name for p in matches)
        raise ValueError(f'Ambiguous caption files: {names}')
    return matches[0]

In [ ]:
# Test: helpers
assert _pick_lang({'en': []}) == 'en'
assert _pick_lang({'live_chat': [], 'en-US': []}) == 'en-US'
assert _pick_lang({'ja': []}) is None

from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    base = Path(d)
    assert _find_downloaded_srt(base) is None
    sample = base / 'captions.en-US.srt'
    sample.write_text('1', encoding='utf-8')
    assert _find_downloaded_srt(base) == sample
print('ok')

In [ ]:
#| export
def get_video_info(url: str # YouTube video URL
                  ) -> dict: # yt-dlp info dict
    "Extract video metadata and caption info without downloading."
    with yt_dlp.YoutubeDL({'skip_download': True, 'quiet': True}) as ydl:
        return ydl.extract_info(url, download=False)

In [ ]:
#| network
# Test: get_video_info smoke test (requires network)
info = get_video_info('https://www.youtube.com/live/Ap4DmaB0gLY')
for k in ['id', 'title', 'channel', 'duration', 'upload_date', 'webpage_url']:
    assert k in info, f'missing {k}'
assert info['id'] == 'Ap4DmaB0gLY'
print('ok')

In [ ]:
#| export
def fetch_video(url: str, # YouTube video URL
                info: dict, # Result of get_video_info
                root: str | Path = None, # Root download directory (default: ~/.cache/yttoc)
               ) -> Path: # Path to video directory
    "Save metadata and English srt captions for one video."
    root = Path(root) if root else _DEFAULT_ROOT
    out_dir = root / info['id']
    out_dir.mkdir(parents=True, exist_ok=True)

    srt_path = out_dir / 'captions.en.srt'
    meta_path = out_dir / 'metadata.json'
    if srt_path.exists() and meta_path.exists():
        return out_dir

    manual_lang = _pick_lang(info.get('subtitles', {}))
    auto_lang = _pick_lang(info.get('automatic_captions', {}))
    selected_lang = manual_lang or auto_lang
    if selected_lang is None:
        raise ValueError(f"No English captions available for {info['id']}")
    caption_type = 'manual' if manual_lang else 'auto'

    meta = {
        'id': info['id'],
        'title': info['title'],
        'channel': info['channel'],
        'duration': info['duration'],
        'upload_date': info['upload_date'],
        'webpage_url': info['webpage_url'],
        'caption_type': caption_type,
    }

    sub_opt = 'writesubtitles' if manual_lang else 'writeautomaticsub'
    opts = {
        'skip_download': True, 'quiet': True,
        sub_opt: True,
        'subtitleslangs': [selected_lang],
        'subtitlesformat': 'srt',
        'outtmpl': str(out_dir / 'captions.%(ext)s'),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    downloaded = _find_downloaded_srt(out_dir)
    if downloaded is None:
        raise FileNotFoundError(f"yt-dlp did not write an srt caption for {info['id']}")
    if downloaded != srt_path:
        downloaded.replace(srt_path)

    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
    return out_dir

In [ ]:
#| eval: false
# Test: fetch_video (requires network)
import shutil
test_root = '/tmp/yttoc_test'
shutil.rmtree(test_root, ignore_errors=True)

out = fetch_video(urls[5], info, root=test_root)
assert (out / 'metadata.json').exists()
assert (out / 'captions.en.srt').exists()

meta = json.loads((out / 'metadata.json').read_text(encoding='utf-8'))
assert meta['id'] == 'Ap4DmaB0gLY'
assert 'series' not in meta  # no longer stored

shutil.rmtree(test_root)
print('ok')

In [ ]:
#| export
def fetch_many(urls: list[str], # List of YouTube URLs
               root: str | Path = None, # Root download directory (default: ~/.cache/yttoc)
              ) -> list[Path]: # List of output Paths
    "Fetch metadata and captions for all videos in a URL list."
    results = []
    for i, url in enumerate(urls):
        print(f'[{i+1}/{len(urls)}] {url}')
        info = get_video_info(url)
        out = fetch_video(url, info, root)
        results.append(out)
    return results

In [ ]:
#| eval: false
# Test: fetch_many (first 2 videos only, requires network)
import shutil
test_root = '/tmp/yttoc_test_many'
shutil.rmtree(test_root, ignore_errors=True)

results = fetch_many(urls[:2], root=test_root)

assert len(results) == 2
for r in results:
    assert (r / 'metadata.json').exists()
    assert (r / 'captions.en.srt').exists()

shutil.rmtree(test_root)
print('ok')

In [ ]:
#| eval: false
# Download all 11 solveit2 videos (requires network)
urls, _ = load_urls('solveit2.txt')
results = fetch_many(urls)
print(f'\nDone: {len(results)} videos')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse

@call_parse
def yttoc_fetch(url: str, # YouTube video URL
                root: str = None, # Root download directory (default: ~/.cache/yttoc)
               ):
    "Fetch metadata and English captions for a single YouTube video."
    info = get_video_info(url)
    out = fetch_video(url, info, root)
    print(out)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()